In [1]:
# ============================================================
# 0) Install ultralytics if not 
# ============================================================
!pip -q install -U ultralytics

from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
#  Model train on train set 1
DATA_DIR = Path("/kaggle/input/multi-view-pig-posture-recognition/multiview_pig_posture_recognition")
OUT_DIR  = Path("/kaggle/working")

train_csv = DATA_DIR / "train1.csv"
test_csv  = DATA_DIR / "test.csv"
train_img_dir = DATA_DIR / "train1_images"
test_img_dir  = DATA_DIR / "test_images"

print(train_csv.exists(), test_csv.exists(), train_img_dir.exists(), test_img_dir.exists())

# ============================================================
# 1) Helpers: bbox parsing + crop
# ============================================================
def parse_bbox_string(bbox_str: str):
    s = str(bbox_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
    s = s.replace(",", " ")
    parts = [p for p in s.split() if p]
    if len(parts) != 4:
        raise ValueError(f"Bad bbox: {bbox_str}")
    x, y, w, h = map(float, parts)
    return x, y, w, h

def safe_crop(img, x, y, w, h, pad=10):
    H, W = img.shape[:2]
    x1 = int(max(0, np.floor(x) - pad))
    y1 = int(max(0, np.floor(y) - pad))
    x2 = int(min(W, np.ceil(x + w) + pad))
    y2 = int(min(H, np.ceil(y + h) + pad))
    if x2 <= x1 or y2 <= y1:
        return np.zeros((32,32,3), dtype=img.dtype)
    return img[y1:y2, x1:x2]

# ============================================================
# 2) Build classification crop dataset from train.csv
# ============================================================
df = pd.read_csv(train_csv)
assert "class_id" in df.columns and "bbox" in df.columns and "image_id" in df.columns and "row_id" in df.columns

# split train/val by row (simple baseline)
seed = 42
rng = np.random.default_rng(seed)
perm = rng.permutation(len(df))
val_ratio = 0.2
n_val = int(round(val_ratio * len(df)))
val_idx = set(perm[:n_val].tolist())

cls_root = OUT_DIR / "cls_data"
train_root = cls_root / "train"
val_root   = cls_root / "val"
train_root.mkdir(parents=True, exist_ok=True)
val_root.mkdir(parents=True, exist_ok=True)

# Create class folders
classes = sorted(df["class_id"].unique().tolist())
for c in classes:
    (train_root / str(c)).mkdir(parents=True, exist_ok=True)
    (val_root / str(c)).mkdir(parents=True, exist_ok=True)

pad = 10
img_size = 224

# Cache images to speed up (optional)
img_cache = {}

def read_rgb(path: Path):
    im = cv2.imread(str(path))
    if im is None:
        raise RuntimeError(f"Cannot read {path}")
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

print("Building crop dataset...")
for i, r in tqdm(df.iterrows(), total=len(df)):
    img_path = train_img_dir / r["image_id"]
    key = str(img_path)
    if key not in img_cache:
        img_cache[key] = read_rgb(img_path)
    img = img_cache[key]

    x, y, w, h = parse_bbox_string(r["bbox"])
    crop = safe_crop(img, x, y, w, h, pad=pad)
    crop = cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_AREA)

    split_root = val_root if i in val_idx else train_root
    out_path = split_root / str(int(r["class_id"])) / f"{r['row_id']}.jpg"

    # Ultralytics expects standard images; save RGB as BGR
    cv2.imwrite(str(out_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))

print("Crop dataset ready:", cls_root)

True True True True
Building crop dataset...


100%|██████████| 22934/22934 [01:46<00:00, 214.72it/s]


Crop dataset ready: /kaggle/working/cls_data
Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cls_data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=Fal

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 3) Train Ultralytics classification model
# ============================================================
# Common options: yolo11n-cls.pt, yolo11s-cls.pt (names can vary by ultralytics version)
# Here we use yolo11n-cls.pt as baseline.
weights = "yolo11n-cls.pt"
try:
    model = YOLO(weights)
except Exception as e:
    print("Failed to load", weights, "-> fallback to yolov8n-cls.pt")
    weights = "yolov8n-cls.pt"
    model = YOLO(weights)

results = model.train(
    data=str(cls_root),   # folder with train/ and val/
    epochs=40,
    imgsz=img_size,
    batch=128,
    device=0,
    workers=2,
    seed=seed,
    
    fliplr=0.0,
    flipud=0.0,
)

best_pt = model.trainer.best
print("Best model:", best_pt)

# ============================================================
# 4) Predict on test.csv crops and build submission.csv
# ============================================================
df_test = pd.read_csv(test_csv)

# Load model from best checkpoint
model = YOLO(best_pt)

test_cache = {}

def get_img(path: Path):
    key = str(path)
    if key not in test_cache:
        test_cache[key] = read_rgb(path)
    return test_cache[key]

pred_rows = []

print("Predicting test crops...")
for _, r in tqdm(df_test.iterrows(), total=len(df_test)):
    img = get_img(test_img_dir / r["image_id"])
    x, y, w, h = parse_bbox_string(r["bbox"])
    crop = safe_crop(img, x, y, w, h, pad=pad)
    crop = cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_AREA)

    # Ultralytics predict accepts numpy image (BGR) or path; we pass BGR
    crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
    out = model.predict(source=crop_bgr, verbose=False)

    # top1 class index
    top1 = int(out[0].probs.top1)
    pred_rows.append((r["row_id"], top1))

sub_path = OUT_DIR / "submission.csv"
pd.DataFrame(pred_rows, columns=["row_id", "class_id"]).to_csv(sub_path, index=False)
print("Wrote:", sub_path)
print(pd.read_csv(sub_path).head())

# ============================================================
# 5) Save important outputs permanently for future
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/output_train1")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save crops for debugging
if (OUT_DIR / "cls_data").exists():
    !cp -r /kaggle/working/cls_data /kaggle/working/output/

# Save best model checkpoint
best_model_out = OUTPUT_DIR / "best.pt"
!cp "{best_pt}" "{best_model_out}"

# Save submission file
submission_out = OUTPUT_DIR / "submission.csv"
!cp /kaggle/working/submission.csv  "{submission_out}"

print("Saved outputs to:", OUTPUT_DIR)
print("Files:", os.listdir(OUTPUT_DIR))

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cls_data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [ ]:
# Model train on train set 2
DATA_DIR = Path("/kaggle/input/multi-view-pig-posture-recognition/multiview_pig_posture_recognition")
OUT_DIR  = Path("/kaggle/working")

train_csv = DATA_DIR / "train2.csv"
test_csv  = DATA_DIR / "test.csv"
train_img_dir = DATA_DIR / "train2_images"
test_img_dir  = DATA_DIR / "test_images"

print(train_csv.exists(), test_csv.exists(), train_img_dir.exists(), test_img_dir.exists())

# ============================================================
# 1) Helpers: bbox parsing + crop
# ============================================================
def parse_bbox_string(bbox_str: str):
    s = str(bbox_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
    s = s.replace(",", " ")
    parts = [p for p in s.split() if p]
    if len(parts) != 4:
        raise ValueError(f"Bad bbox: {bbox_str}")
    x, y, w, h = map(float, parts)
    return x, y, w, h

def safe_crop(img, x, y, w, h, pad=10):
    H, W = img.shape[:2]
    x1 = int(max(0, np.floor(x) - pad))
    y1 = int(max(0, np.floor(y) - pad))
    x2 = int(min(W, np.ceil(x + w) + pad))
    y2 = int(min(H, np.ceil(y + h) + pad))
    if x2 <= x1 or y2 <= y1:
        return np.zeros((32,32,3), dtype=img.dtype)
    return img[y1:y2, x1:x2]

# ============================================================
# 2) Build classification crop dataset from train.csv
# ============================================================
df = pd.read_csv(train_csv)
assert "class_id" in df.columns and "bbox" in df.columns and "image_id" in df.columns and "row_id" in df.columns

# split train/val by row (simple baseline)
seed = 42
rng = np.random.default_rng(seed)
perm = rng.permutation(len(df))
val_ratio = 0.2
n_val = int(round(val_ratio * len(df)))
val_idx = set(perm[:n_val].tolist())

cls_root = OUT_DIR / "cls_data"
train_root = cls_root / "train"
val_root   = cls_root / "val"
train_root.mkdir(parents=True, exist_ok=True)
val_root.mkdir(parents=True, exist_ok=True)

# Create class folders
classes = sorted(df["class_id"].unique().tolist())
for c in classes:
    (train_root / str(c)).mkdir(parents=True, exist_ok=True)
    (val_root / str(c)).mkdir(parents=True, exist_ok=True)

pad = 10
img_size = 224

# Cache images to speed up (optional)
img_cache = {}

def read_rgb(path: Path):
    im = cv2.imread(str(path))
    if im is None:
        raise RuntimeError(f"Cannot read {path}")
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

print("Building crop dataset...")
for i, r in tqdm(df.iterrows(), total=len(df)):
    img_path = train_img_dir / r["image_id"]
    key = str(img_path)
    if key not in img_cache:
        img_cache[key] = read_rgb(img_path)
    img = img_cache[key]

    x, y, w, h = parse_bbox_string(r["bbox"])
    crop = safe_crop(img, x, y, w, h, pad=pad)
    crop = cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_AREA)

    split_root = val_root if i in val_idx else train_root
    out_path = split_root / str(int(r["class_id"])) / f"{r['row_id']}.jpg"

    # Ultralytics expects standard images; save RGB as BGR
    cv2.imwrite(str(out_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))

print("Crop dataset ready:", cls_root)

# ============================================================
# 3) Train Ultralytics classification model
# ============================================================
# Common options: yolo11n-cls.pt, yolo11s-cls.pt (names can vary by ultralytics version)
# Here we use yolo11n-cls.pt as baseline.
weights = "yolo11n-cls.pt"
try:
    model = YOLO(weights)
except Exception as e:
    print("Failed to load", weights, "-> fallback to yolov8n-cls.pt")
    weights = "yolov8n-cls.pt"
    model = YOLO(weights)

results = model.train(
    data=str(cls_root),   # folder with train/ and val/
    epochs=40,
    imgsz=img_size,
    batch=64,
    device=0,
    workers=2,
    seed=seed,
    
    fliplr=0.0,
    flipud=0.0,
)

best_pt = model.trainer.best
print("Best model:", best_pt)

# ============================================================
# 4) Predict on test.csv crops and build submission.csv
# ============================================================
df_test = pd.read_csv(test_csv)

# Load model from best checkpoint
model = YOLO(best_pt)

test_cache = {}

def get_img(path: Path):
    key = str(path)
    if key not in test_cache:
        test_cache[key] = read_rgb(path)
    return test_cache[key]

pred_rows = []

print("Predicting test crops...")
for _, r in tqdm(df_test.iterrows(), total=len(df_test)):
    img = get_img(test_img_dir / r["image_id"])
    x, y, w, h = parse_bbox_string(r["bbox"])
    crop = safe_crop(img, x, y, w, h, pad=pad)
    crop = cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_AREA)

    # Ultralytics predict accepts numpy image (BGR) or path; we pass BGR
    crop_bgr = cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
    out = model.predict(source=crop_bgr, verbose=False)

    # top1 class index
    top1 = int(out[0].probs.top1)
    pred_rows.append((r["row_id"], top1))

sub_path = OUT_DIR / "submission.csv"
pd.DataFrame(pred_rows, columns=["row_id", "class_id"]).to_csv(sub_path, index=False)
print("Wrote:", sub_path)
print(pd.read_csv(sub_path).head())

# ============================================================
# 5) Save important outputs permanently for future
# ============================================================

OUTPUT_DIR = Path("/kaggle/working/output_train2")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save crops for debugging
if (OUT_DIR / "cls_data").exists():
    !cp -r /kaggle/working/cls_data /kaggle/working/output/

# Save best model checkpoint
best_model_out = OUTPUT_DIR / "best.pt"
!cp "{best_pt}" "{best_model_out}"

# Save submission file
submission_out = OUTPUT_DIR / "submission.csv"
!cp /kaggle/working/submission.csv  "{submission_out}"

print("Saved outputs to:", OUTPUT_DIR)
print("Files:", os.listdir(OUTPUT_DIR))